In [1]:
import sys, json
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestRegressor

sys.path.append("..")
from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"train={len(train):,} | val={len(val):,} | test={len(test):,}")

train=269,112 | val=3,926 | test=3,872


In [3]:
print(test[0].summary)

Tiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  
Danh mục: Hộp đựng, Thùng lưu trữ  
Thương hiệu: Index Living Mall  
Mô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  
Thông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.


In [8]:
from litellm import completion

In [19]:
def messages_for(item):
    message = f"Hãy ước lượng giá của sản phẩm này.\
 Chỉ phản hồi lại mức giá, không giải thích gì thêm\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [21]:
def messages_for(item):
    message = f"Hãy ước lượng giá của sản phẩm này. Chỉ trả về chính xác một con số trong khoảng từ 5 đến 1000 (Ví dụ: 350 tương ứng với 350.000 đồng), không giải thích gì thêm\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [22]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Hãy ước lượng giá của sản phẩm này. Chỉ trả về chính xác một con số trong khoảng từ 5 đến 1000 (Ví dụ: 350 tương ứng với 350.000 đồng), không giải thích gì thêm\n\nTiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.'}]

In [23]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [24]:
gpt_4__1_nano(test[0])

'250'

In [25]:
test[0].price

479

In [26]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

329 51 124 10 29 605 134 141 100 88 119 5 377 98 294 39 20 101 140 427 84 600 23 15 250 219 76 35 665 131 21 71 107 31 74 61 210 19 714 655 145 74 200 182 160 430 0 15 64 1 87 15 62 29 375 69 297 46 125 200 575 70 88 165 50 25 9 266 350 45 15 131 39 200 61 94 200 281 1 71 95 340 100 390 171 158 201 151 141 51 299 351 3 240 51 61 55 765 171 310 60 29 108 51 113 68 70 112 160 130 293 61 549 80 305 10 29 499 46 35 291 208 21 16 456 121 39 29 604 80 280 2 454 95 68 11 199 424 182 120 30 123 100 95 175 65 451 219 5 285 298 97 649 249 33 549 11 349 21 82 80 181 85 83 121 82 1 219 126 740 278 5 354 65 630 240 395 95 270 387 75 19 0 115 43 60 21 468 49 15 375 289 25 49 111 57 61 670 31 80 


{'mae': 170.76, 'mse': 59997.44, 'r2': -4.507555282392106}

In [28]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="openai/gpt-5.1", messages=messages_for(item))
    return response.choices[0].message.content

In [32]:
gpt_5__1(test[2])

'75'

In [33]:
test[2].price

26

In [31]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

259 81 19 90 6 165 36 51 35 28 51 116 307 7 144 39 35 71 10 97 114 300 97 120 220 79 76 5 495 1 9 1 52 131 144 31 10 49 474 215 125 35 200 82 90 360 5 45 21 61 12 75 162 229 105 40 3 116 95 170 375 100 7 104 80 45 61 96 160 45 45 101 39 130 31 1 30 14 89 119 25 370 71 70 216 128 171 21 111 60 129 221 107 570 121 131 95 480 101 40 10 14 18 121 243 98 170 107 30 40 123 51 349 90 135 390 199 429 19 20 191 78 9 21 196 49 131 24 574 40 250 17 279 65 162 71 161 249 152 10 91 28 30 24 105 65 121 51 10 115 228 67 449 49 33 279 81 51 51 52 140 251 90 17 61 52 21 49 21 570 8 75 21 235 330 80 325 30 40 167 45 49 10 25 27 30 141 398 21 85 220 119 5 21 10 63 31 400 91 1 


{'mae': 114.755, 'mse': 27699.175, 'r2': 51.751723697058516}